In [8]:
!python -m pip install h2o

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.0/266.0 MB 4.0 MB/s eta 0:00:00


In [10]:
!sudo apt update
!sudo apt install default-jre -y
!sudo apt install default-jdk -y

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:2 https://cli.github.com/packages stable InRelease
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,123 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,825 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,526 kB]
Hit:13 https://ppa.launchpadcontent.net/graphics-drivers

In [12]:
import h2o
from h2o.automl import H2OAutoML
import pandas as pd

h2o.init()

file_path = "/content/diabetes.csv"
df = pd.read_csv(file_path)

print("Original Dataset:")
print(df.head())

if 'PatientID' in df.columns:
    df = df.drop(columns=['PatientID'])

h2o_df = h2o.H2OFrame(df)

predictors = list(df.columns[:-1])
response = df.columns[-1]

h2o_df[response] = h2o_df[response].asfactor()

train, test = h2o_df.split_frame(ratios=[0.8], seed=42)

aml = H2OAutoML(
    max_runtime_secs=300,
    seed=42
)
aml.train(x=predictors, y=response, training_frame=train)

print("AutoML Leaderboard:")
print(aml.leaderboard)

leader_model = aml.leader

predictions = leader_model.predict(test)

print("Predictions on the Test Set:")
print(predictions)

model_path = h2o.save_model(model=leader_model, path="./", force=True)
print(f"Best model saved to {model_path}")

h2o.shutdown(prompt=False)

Checking whether there is an H2O instance running at http://localhost:54321. connected.


H2O_cluster_uptime:,1 min 46 secs
H2O_cluster_timezone:,Etc/UTC
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.8
H2O_cluster_version_age:,1 month and 1 day
H2O_cluster_name:,H2O_from_python_unknownUser_0xihyb
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,3.168 Gb
H2O_cluster_total_cores:,2
H2O_cluster_allowed_cores:,2
H2O_cluster_status:,"locked, healthy"


Original Dataset:
   PatientID  Pregnancies  PlasmaGlucose  DiastolicBloodPressure  \
0    1354778            0            171                      80   
1    1147438            8             92                      93   
2    1640031            7            115                      47   
3    1883350            9            103                      78   
4    1424119            1             85                      59   

   TricepsThickness  SerumInsulin        BMI  DiabetesPedigree  Age  Diabetic  
0                34            23  43.509726          1.213191   21         0  
1                47            36  21.240576          0.158365   23         0  
2                52            35  41.511523          0.079019   23         0  
3                25           304  29.582192          1.282870   43         1  
4                27            35  42.604536          0.549542   22         0  
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100

/tmp/ipython-input-4078302771.py:45: H2ODeprecationWarning: Deprecated, use ``h2o.cluster().shutdown()``.
  h2o.shutdown(prompt=False)
